# 01 - Clean corpus (dedup + descriptives)

Builds the cleaned, deduplicated analysis datasets and the corpus descriptives.

**Inputs** (`analysis_data/`): `df_actors_socialdiv_analysis.csv` (article-level, one row
per article, clean `brand`/`type`/`topic`/`country`/`date`) and
`df_actors_politicaldiv_analysis.csv` (actor-level, one row per political actor). Body text
comes from the per-country topic files (`*_df*.csv`) and is used *only* for near-duplicate
detection.

**Outputs**: `*_clean.csv` for both dimensions (identical physical-article set),
`Table3_corpus_deduplicated.csv`, and a corpus funnel.

Shared constants/paths/helpers live in `helpers.py` (imported as `h`) so every notebook
reads the same country order, labels, thresholds and parse rules.

Steps: (1) load social df + attach text, (2) pre-dedup descriptives, (3) scan near-duplicate
pairs, (4) **threshold revisit**, (5) deduplicate (keep most-actor copy), (6) clean the
political df to the same article set (recovering orphan country), (7) export, (8) Table 3 +
funnel.

In [ ]:
import re

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components

import helpers as h

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 70)

print("Project root :", h.BASE_DIR)
print("Dedup cutoff : sim >=", h.TEXT_SIM_THRESHOLD, "| same_brand_only =", h.SAME_BRAND_ONLY)

## 1. Load the social df and attach body text

The social df already carries clean `brand`, `type`, `topic`, `country` and `date`, so no
brand normalisation is needed here. It has no article body, so we pull `text` from the topic
files purely to detect near-duplicates. `parse_dates_safe` re-validates the dates: it raises
(rather than silently coercing to NaT) if any unexpected date format slipped in, the failure
mode that previously dropped 331 Irish RTÉ campaign articles.

In [ ]:
soc = pd.read_csv(h.SOCIAL_RAW, sep=h.CSV_SEP)
assert soc["article_id"].is_unique, "expected one row per article in the social df"

_ = h.parse_dates_safe(soc["date"])   # guard: raises on any un-parseable date, never coerces

soc["has_actor"] = soc["nr_actors_total"].notna()
soc["actors"]    = soc["nr_actors_total"].fillna(0)
print(f"{len(soc):,} articles | {int(soc['actors'].sum()):,} actors | "
      f"{100*(1 - soc['has_actor'].mean()):.1f}% zero-actor")

In [ ]:
soc.head(3) 

In [ ]:
texts = (h.load_topic_files(with_text=True)[["article_id", "text"]]
           .drop_duplicates("article_id"))
soc = soc.merge(texts, on="article_id", how="left")

n_missing = int(soc["text"].isna().sum())
print(f"Body text attached | {n_missing:,} articles without text "
      f"({100*n_missing/len(soc):.1f}%) -> treated as singletons in dedup")

soc["x_norm"] = soc["text"].fillna("").map(h.norm_text)
soc = soc.reset_index(drop=True)   # positional index is used by the pair graph below

In [ ]:
soc.head(3)

## 2. Corpus descriptives (before dedup)

In [ ]:
g = soc.groupby("country")
actor_country = pd.DataFrame({
    "articles":    g.size(),
    "actors":      g["actors"].sum().astype(int),
    "mean_actors": g["actors"].mean().round(2),
    "pct_zero":    (100*(1 - g["has_actor"].mean())).round(1),
}).reindex(h.COUNTRY_ORDER)
actor_country.loc["Total"] = [len(soc), int(soc["actors"].sum()),
                              round(soc["actors"].mean(), 2),
                              round(100*(1 - soc["has_actor"].mean()), 1)]
actor_country

In [ ]:
gbt = soc.groupby(["country", "brand", "topic"])
actor_brand_topic = (pd.DataFrame({
        "articles":    gbt.size(),
        "actors":      gbt["actors"].sum().astype(int),
        "mean_actors": gbt["actors"].mean().round(2),
        "pct_zero":    (100*(1 - gbt["has_actor"].mean())).round(1),
    }).reset_index()
      .sort_values(["country", "brand", "topic"]))
h.RESULTS_DIR.mkdir(exist_ok=True)
actor_brand_topic.to_csv(h.TABLES_DIR / "actor_brand_topic_summary.csv", index=False)
actor_brand_topic.head(12)

## 3. Scan candidate near-duplicate pairs

Collect every within-country text pair with cosine similarity >= `SCAN_THRESHOLD` (0.90)
**once**. We then re-filter cheaply at any decision cutoff without recomputing. Dedup is
within-country by construction: cross-country duplicates are not meaningful.

In [ ]:
NEARDUP_BLOCK = 2000

def scan_pairs(global_idx, texts, thresh, block=NEARDUP_BLOCK):
    """All within-block pairs (i<j) with cosine similarity >= thresh.
    Returns global soc positions (gi, gj) and the similarity."""
    vec = TfidfVectorizer(min_df=2, max_df=0.5, sublinear_tf=True,
                          max_features=60000, dtype=np.float32)
    X = vec.fit_transform(texts); n = X.shape[0]; Xt = X.T.tocsr()
    GI, GJ, V = [], [], []
    for s in range(0, n, block):
        sim = (X[s:s+block] @ Xt)
        sim.data[sim.data < thresh] = 0; sim.eliminate_zeros()
        sim = sim.tocoo()
        li, lj, lv = sim.row + s, sim.col, sim.data
        keep = li < lj
        GI.append(global_idx[li[keep]]); GJ.append(global_idx[lj[keep]]); V.append(lv[keep])
    if GI and sum(len(x) for x in GI):
        return np.concatenate(GI), np.concatenate(GJ), np.concatenate(V)
    return np.array([], int), np.array([], int), np.array([], float)

parts = []
for cc in h.COUNTRY_ORDER:
    idx = soc.index[soc["country"] == cc].to_numpy()
    gi, gj, v = scan_pairs(idx, soc.loc[idx, "x_norm"].tolist(), h.SCAN_THRESHOLD)
    if len(gi):
        parts.append(pd.DataFrame({"country": cc, "gi": gi, "gj": gj, "sim": v}))
    print(f"{cc}: {len(gi):,} candidate pairs >= {h.SCAN_THRESHOLD}")

pairs = pd.concat(parts, ignore_index=True)
pairs["brand_a"] = soc.loc[pairs["gi"], "brand"].to_numpy()
pairs["brand_b"] = soc.loc[pairs["gj"], "brand"].to_numpy()
pairs["same_brand"] = pairs["brand_a"] == pairs["brand_b"]
print(f"\nTotal candidate pairs: {len(pairs):,}")

## 4. Threshold revisit

Calibrate the decision cutoff (`TEXT_SIM_THRESHOLD`, currently 0.98) and the
`same_brand_only` rule. First the distribution of candidate pairs by similarity band and
same/cross brand; then example pairs around the cutoff, so we can see where genuine
duplicates end and merely-similar articles begin. Adjust the values in `helpers.py` and
re-run from here; the expensive scan above does not rerun.

In [ ]:
bands  = [0.85, 0.90, 0.93, 0.95, 0.97, 0.99, 1.0001]
labels = ["0.85-0.90", "0.90-0.93", "0.93-0.95", "0.95-0.97", "0.97-0.99", "0.99-1.00"]
pairs["band"] = pd.cut(pairs["sim"], bins=bands, labels=labels, right=False)
dist = pairs.pivot_table(index="band", columns="same_brand", values="sim",
                         aggfunc="count", fill_value=0, observed=False)
dist.columns = ["cross_brand" if c is False else "same_brand" for c in dist.columns]
dist["total"] = dist.sum(axis=1)
dist

In [ ]:
def show_pairs_in_band(lo, hi, n=4, chars=300, seed=0, same_brand=None):
    band = pairs[(pairs["sim"] >= lo) & (pairs["sim"] < hi)]
    if same_brand is not None:
        band = band[band["same_brand"] == same_brand]
    print(f"{len(band):,} pairs in [{lo}, {hi})"
          + ("" if same_brand is None else f"  (same_brand={same_brand})"))
    for _, r in band.sample(min(n, len(band)), random_state=seed).iterrows():
        a, b = soc.loc[r["gi"]], soc.loc[r["gj"]]
        print("=" * 100)
        print(f"sim={r['sim']:.3f} | {r['country']} | same_brand={r['same_brand']}")
        print(f"[A] {a['brand']:>20} | actors={a['actors']:>4.0f} | {str(a['title'])[:70]}")
        print("     ", h.norm_text(a["text"])[:chars])
        print(f"[B] {b['brand']:>20} | actors={b['actors']:>4.0f} | {str(b['title'])[:70]}")
        print("     ", h.norm_text(b["text"])[:chars])

# Below the cutoff (should still be clearly NOT duplicates) and at/above it.
show_pairs_in_band(0.95, 0.98, n=3, seed=0)
show_pairs_in_band(0.98, 1.0001, n=3, seed=0)

In [ ]:
def build_text_clusters(pairs_df, threshold, same_brand_only, n_rows):
    p = pairs_df[pairs_df["sim"] >= threshold]
    if same_brand_only:
        p = p[p["same_brand"]]
    cl = np.full(n_rows, -1)
    if len(p):
        adj = csr_matrix((np.ones(len(p)), (p["gi"].to_numpy(), p["gj"].to_numpy())),
                         shape=(n_rows, n_rows))
        adj = adj + adj.T
        _, lab = connected_components(adj, directed=False)
        sizes = pd.Series(lab).value_counts()
        multi = sizes[sizes > 1].index
        mask = np.isin(lab, multi)
        cl[mask] = lab[mask]
    return cl

def dedup_keep_most_actors(df):
    """Collapse each text-duplicate cluster to one article: most actors, then longer
    text, then lowest id. Singletons (cluster < 0) pass through untouched."""
    df = df.copy()
    df["_tie_len"] = df["x_norm"].str.len()
    singletons = df[df["text_cluster"] < 0]
    winners = (df[df["text_cluster"] >= 0]
               .sort_values(["text_cluster", "actors", "_tie_len", "article_id"],
                            ascending=[True, False, False, True])
               .groupby("text_cluster", as_index=False).head(1))
    kept = pd.concat([singletons, winners], ignore_index=True).drop(columns="_tie_len")
    removed = df[~df["article_id"].isin(kept["article_id"])]
    return kept, removed

# Sensitivity: how removal and the headline descriptives move with the cutoff.
rows = []
for thr in [0.93, 0.95, 0.97, 0.98]:
    cl = build_text_clusters(pairs, thr, h.SAME_BRAND_ONLY, len(soc))
    kept, removed = dedup_keep_most_actors(soc.assign(text_cluster=cl))
    rows.append({"threshold": thr, "removed": len(removed),
                 "pct_removed": round(100*len(removed)/len(soc), 1),
                 "articles_after": len(kept),
                 "mean_actors_after": round(kept["actors"].mean(), 3),
                 "pct_zero_after": round(100*(1 - kept["has_actor"].mean()), 2)})
print(f"baseline (no dedup): {len(soc):,} articles | mean actors "
      f"{soc['actors'].mean():.3f} | %zero {100*(1-soc['has_actor'].mean()):.2f}\n")
pd.DataFrame(rows).set_index("threshold")

## 5. Deduplicate at the chosen cutoff = > 0.98

Build clusters at `TEXT_SIM_THRESHOLD` / `SAME_BRAND_ONLY` and collapse each to the
article with the most actors. This is the analysis set.

In [ ]:
TEXT_SIM_THRESHOLD = 0.98   # DECISION cutoff: text pairs with cosine >= this are duplicates

soc["text_cluster"] = build_text_clusters(pairs, h.TEXT_SIM_THRESHOLD,
                                          h.SAME_BRAND_ONLY, len(soc))
soc_dedup, soc_removed = dedup_keep_most_actors(soc)

n_clusters  = int(soc.loc[soc["text_cluster"] >= 0, "text_cluster"].nunique())
n_redundant = int((soc["text_cluster"] >= 0).sum() - n_clusters)
print(f"cutoff sim >= {TEXT_SIM_THRESHOLD}, same_brand_only={h.SAME_BRAND_ONLY}")
print(f"duplicate clusters : {n_clusters:,}")
print(f"redundant articles : {n_redundant:,} ({100*n_redundant/len(soc):.1f}% of corpus)")
print(f"articles           : {len(soc):,} -> {len(soc_dedup):,} (removed {len(soc_removed):,})")

In [ ]:
compare = pd.DataFrame({
    "before": soc.groupby("country").size().reindex(h.COUNTRY_ORDER),
    "after":  soc_dedup.groupby("country").size().reindex(h.COUNTRY_ORDER),
})
compare["removed"] = compare["before"] - compare["after"]
compare["pct_removed"] = (100*compare["removed"]/compare["before"]).round(1)
compare.loc["Total"] = [compare["before"].sum(), compare["after"].sum(),
                        compare["removed"].sum(),
                        round(100*compare["removed"].sum()/compare["before"].sum(), 1)]
compare

## 6. Clean the political df to the same article set

The political df is actor-level. We keep an actor-row if its article survived the social
dedup, so both cleaned files reference the identical physical articles. **Orphans** (political
articles absent from the social df, hence no country and no dedup decision) keep their
`article_id`; we recover their country from its leading digit (1=UK, 2=IR, 3=NL, 4=BE, 5=TR)
and retain them.

In [ ]:
pol = pd.read_csv(h.POLITICAL_RAW, sep=h.CSV_SEP)

soc_ids  = set(soc["article_id"])
kept_ids = set(soc_dedup["article_id"])

orphan  = ~pol["article_id"].isin(soc_ids)
need_cc = orphan & pol["country"].isna()
pol.loc[need_cc, "country"] = pol.loc[need_cc, "article_id"].map(h.country_from_id)
print(f"orphan rows: {int(orphan.sum()):,} "
      f"({pol.loc[orphan, 'article_id'].nunique():,} articles) | "
      f"country recovered for {int(pol.loc[orphan, 'country'].notna().sum()):,} rows")

# keep social-kept representatives AND orphans (orphans are not part of the dedup scan)
keep = pol["article_id"].isin(kept_ids) | orphan
pol_clean = pol[keep].copy()
print(f"actor-rows: {len(pol):,} -> {len(pol_clean):,} | "
      f"articles: {pol['article_id'].nunique():,} -> {pol_clean['article_id'].nunique():,}")

In [ ]:
pol.head(3)

In [ ]:
pol_compare = pd.DataFrame({
    "articles_before": pol.groupby("country")["article_id"].nunique().reindex(h.COUNTRY_ORDER),
    "articles_after":  pol_clean.groupby("country")["article_id"].nunique().reindex(h.COUNTRY_ORDER),
    "rows_before":     pol.groupby("country").size().reindex(h.COUNTRY_ORDER),
    "rows_after":      pol_clean.groupby("country").size().reindex(h.COUNTRY_ORDER),
}).fillna(0).astype(int)
pol_compare.loc["Total"] = pol_compare.sum()
pol_compare

## 7. Export cleaned datasets

In [ ]:
# Drop working columns; keep the analysis columns + dedup provenance (text_cluster).
work_cols = ["text", "x_norm", "has_actor", "actors"]
soc_out = soc_dedup.drop(columns=work_cols, errors="ignore")
soc_out.to_csv(h.SOCIAL_CLEAN, sep=h.CSV_SEP, index=False)
pol_clean.to_csv(h.POLITICAL_CLEAN, sep=h.CSV_SEP, index=False)
print("wrote", h.SOCIAL_CLEAN.name, f"({len(soc_out):,} articles)")
print("wrote", h.POLITICAL_CLEAN.name, f"({len(pol_clean):,} actor-rows)")

## 8. Table 3 and corpus funnel

Table 3 is the per country x outlet x issue corpus description on the deduplicated set.
The funnel covers on-topic -> relevant -> in-social -> after-dedup; the raw "collected"
counts (Table 2) come from the external retrieval files and are produced separately.

In [ ]:
def build_table3(df):
    df = df.copy()
    df["actors"]  = df["nr_actors_total"].fillna(0)
    df["is_zero"] = df["nr_actors_total"].isna()

    def stats(d):
        n = len(d); act = int(d["actors"].sum())
        return {"Immigration": int((d["topic"] == "immigration").sum()),
                "Inflation":   int((d["topic"] == "rising_prices").sum()),
                "Articles": n, "Actors": act,
                "M_actors_article": round(act/n, 2) if n else 0,
                "Pct_zero": round(100*d["is_zero"].mean(), 1) if n else 0,
                "M_words": round(d["word_count"].mean(), 0) if n else 0}

    rows = []
    for cc in h.COUNTRY_ORDER:
        sub = df[df["country"] == cc]
        for t in h.TYPE_ORDER:
            st = sub[sub["type"] == t]
            order = sorted(st["brand"].unique(),
                           key=lambda b: (st.loc[st["brand"] == b, "left_leaning"].mean()
                                          if t == "Quality newspaper" else 0),
                           reverse=True)
            for b in order:
                rows.append({"Country": h.COUNTRY_NAME[cc], "Type": h.TYPE_LABEL[t],
                             "Outlet": h.BRAND_DISP.get(b, b), **stats(st[st["brand"] == b])})
        rows.append({"Country": h.COUNTRY_NAME[cc], "Type": "", "Outlet": "Subtotal", **stats(sub)})
    rows.append({"Country": "", "Type": "", "Outlet": "Total", **stats(df)})
    return pd.DataFrame(rows)

table3 = build_table3(soc_dedup)
table3.to_csv(h.TABLES_DIR / "Table3_corpus_deduplicated.csv", index=False)
print("saved Table3_corpus_deduplicated.csv | cutoff sim >=", h.TEXT_SIM_THRESHOLD)
table3

In [ ]:
topic = h.load_topic_files(with_text=False)
funnel = pd.DataFrame(index=h.COUNTRY_ORDER)
funnel["on_topic"]    = (topic.drop_duplicates("article_id")
                              .groupby("country").size().reindex(h.COUNTRY_ORDER))
funnel["relevant"]    = (topic[topic["relevant"] == 1].drop_duplicates("article_id")
                              .groupby("country").size().reindex(h.COUNTRY_ORDER))
funnel["in_social"]   = soc.groupby("country").size().reindex(h.COUNTRY_ORDER)
funnel["after_dedup"] = soc_dedup.groupby("country").size().reindex(h.COUNTRY_ORDER)
funnel.loc["Total"] = funnel.sum()
funnel.astype("Int64")